# Predicting the Harvard-Yale Game with Least Squares

**Goal:** fit a scoring model for Yale and one for Harvard on 2016-2025 Ivy League game data, then use each team's current-season form to predict the score of the upcoming Harvard-Yale game.

Each model predicts points scored as:

```
points = c0 + c1*off_ypp + c2*off_3d + c3*off_to + c4*def_ypp + c5*def_3d + c6*def_tk + c7*home
```

`off_*` describes the team's own offense; `def_*` describes the opponent's defense, so each row is a matchup, not a single team. Every stat is a season-to-date average computed only from games played *before* the game being modeled, so no game predicts itself.

With more games than unknowns, `M c = y` is overdetermined, so `c` is chosen to minimize `‖M c − y‖` by solving the normal equations:

```
(M^T M) c = M^T y
```

## Input data (`df`)

One row per team-game (each game appears twice, once per team):

| Column | Meaning |
| --- | --- |
| `team`, `opponent` | team names as used in the filters, e.g. `yale`, `harvard` |
| `season`, `date` | season year, and the game date as a sortable `YYYY-MM-DD` string |
| `home` | 1 if `team` played at home, 0 otherwise |
| `team_pts` | points scored by `team`, the quantity being modeled |
| `team_total_yds`, `team_total_plays` | offensive yards and play count for `team` |
| `team_third_down_conv`, `team_third_down_att` | third downs converted and attempted by `team` |
| `team_turnovers` | turnovers committed by `team` |
| `opp_total_yds`, `opp_total_plays` | yards and plays allowed by the defense of `team` |
| `opp_third_down_conv`, `opp_third_down_att` | third downs allowed and faced by the defense of `team` |
| `opp_turnovers` | turnovers forced by the defense of `team` |

## Imports

`pandas` loads and filters the game log; `numpy` solves the normal equations.

In [22]:
import pandas as pd
import numpy as np

## Building the least-squares system

- **`average(date, season, team)`** — season-to-date offense (`off_ypp`, `off_3d`, `off_to`) and defense (`def_ypp`, `def_3d`, `def_tk`) for `team`, using only games before `date`. Returns `None` for season openers (no prior games).
- **`buildrow(date, team)`** — one game as a row of `M`: `[1, own offense..., opponent defense..., home]`, paired with `team_pts` as the matching entry of `y`.
- **`buildmatrix(team)`** — stacks every valid row/points pair across the team's history into `M` and `b`.
- **`calc(team)`** — solves `(M^T M) c = M^T y` for `c`, rounded to 2 decimals.

In [ ]:
df = pd.read_csv("data/ivy_team_games_2016_2025.csv")

def average(date, season, team):
    # only games before `date` this season, so no game predicts itself
    games = df[(df["team"] == team) & (df["season"] == season) & (df["date"] < date)]

    if len(games) == 0:
        return None

    stats = {}

    stats["off_ypp"] = games["team_total_yds"].sum() / games["team_total_plays"].sum()
    stats["off_3d"]  = games["team_third_down_conv"].sum() / games["team_third_down_att"].sum()
    stats["off_to"]  = games["team_turnovers"].sum() / len(games)

    stats["def_ypp"] = games["opp_total_yds"].sum() / games["opp_total_plays"].sum()
    stats["def_3d"]  = games["opp_third_down_conv"].sum() / games["opp_third_down_att"].sum()
    stats["def_tk"]  = games["opp_turnovers"].sum() / len(games)

    return stats

def buildrow(date, team):
    game = df[(df["date"] == date) & (df["team"] == team)]

    if (len(game) != 1):
        return None
    
    team_stats = average(date, game.iloc[0]["season"], team)
    opp_stats = average(date, game.iloc[0]["season"], game.iloc[0]["opponent"])
    home = game.iloc[0]["home"]

    if (team_stats is None or opp_stats is None):
        return None

    row = [1, team_stats["off_ypp"], team_stats["off_3d"], team_stats["off_to"], opp_stats["def_ypp"], opp_stats["def_3d"], opp_stats["def_tk"], home]
    points = game.iloc[0]["team_pts"]

    return [row, points]

def buildmatrix(team):
    games = df[(df["team"] == team)]

    M = []
    b = []

    for game in games.itertuples():
        currRow = buildrow(game.date, team)

        if (currRow is None):
            continue

        M.append(currRow[0])
        b.append(currRow[1])

    return [M, b]

def calc(team):
    input = buildmatrix(team)
    M = np.array(input[0])
    M_T = np.transpose(M)
    y = np.array(input[1])
    
    # solve the normal equations (M^T M) c = M^T y for the least-squares coefficients
    return np.round(np.linalg.solve(np.matmul(M_T, M), np.matmul(M_T, y)), 2)

## Fitting the two models

`calc` returns coefficients in the same order as `buildrow`'s row:

```
[intercept, off_ypp, off_3d, off_to, def_ypp, def_3d, def_tk, home]
```

The predictors are on very different scales (yards/play ≈ 5, rates ∈ [0,1], turnovers/game ≈ 1-2), so coefficient size isn't directly comparable across terms. (`# Models`, `## Yale`, `## Harvard` below are plain comments, not rendered headings.)

In [24]:
# Models

## Yale

ycoeff = calc("yale")

print(f"Yale Coefficients: {ycoeff}")

## Harvard

hcoeff = calc("harvard")

print(f"Harvard Coefficients: {hcoeff}")

Yale Coefficients: [-14.1    0.94  23.42  -4.09   6.29   5.93  -2.46   8.03]
Harvard Coefficients: [37.45 -5.66 31.64 -8.39  2.21 29.13 -1.95 -1.32]


## Next Steps

- [ ] **Backtest on 2016-2025** — compare each model's predicted points to actual `team_pts` for the games it was fit on (residuals, MAE/RMSE, R²) to see how well it explains the training data.
- [ ] **Validate on 2026** — score the same models against 2026 games as they're played, data the model has never seen, to check it generalizes rather than overfitting to 2016-2025.
- [ ] **Predict the Harvard-Yale game** — once validated, feed each team's current 2026 season-to-date stats (and home/away) into `buildrow`-style inputs to predict the score of the upcoming matchup.